In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
import tensorflow as tf

CAMINHO = "/content/drive/MyDrive/Assistente de Inteligência Artificial - FATESG/Aula - 09"

arquivo = CAMINHO + "/datasetExamesSangue10K.csv"

datasetExames = pd.read_csv(arquivo)
print("Dataset carregado com sucesso!")

print(datasetExames.head())

print("\nQuantidade por diagnóstico:")
print(datasetExames["diagnostico"].value_counts())

dadosEntrada = datasetExames[[
        "idade","hemoglobina","hematocrito","hemacias","leucocitos",
        "plaquetas","glicemia","pressaoSistolica","pressaoDiastolica"]]

dadosSaidaTexto = datasetExames["diagnostico"]
encoderDiagnostico = LabelEncoder()
dadosSaidaNumero = encoderDiagnostico.fit_transform(dadosSaidaTexto)
dadosSaida = to_categorical(dadosSaidaNumero)
print("\nClasses do modelo:")
print(encoderDiagnostico.classes_)

Dataset carregado com sucesso!
   idade  hemoglobina  hematocrito  hemacias  leucocitos  plaquetas  glicemia  \
0     69         16.3         45.1      5.16        4966        394        85   
1     41         15.1         37.6      5.36        6933        371        95   
2     75         14.6         41.8      4.67        5582        349        90   
3     79         13.3         42.7      5.15        7702        408        93   
4     56         12.6         47.4      5.10        6541        188        95   

   pressaoSistolica  pressaoDiastolica diagnostico  
0               110                 68    Saudavel  
1               100                 76    Saudavel  
2               114                 79    Saudavel  
3               106                 73    Saudavel  
4               101                 84    Saudavel  

Quantidade por diagnóstico:
diagnostico
Saudavel       250
Diabetes       250
Leucemia       250
Hipertensao    250
Name: count, dtype: int64

Classes do modelo:
[

In [ ]:
dadosTreinoEntrada, dadosTesteEntrada,dadosTreinoSaida, dadosTesteSaida = train_test_split(dadosEntrada,
    dadosSaida, test_size=0.20, random_state=42)

normalizador = StandardScaler()
dadosTreinoEntradaNormalizados = normalizador.fit_transform(dadosTreinoEntrada)
dadosTesteEntradaNormalizados = normalizador.transform(dadosTesteEntrada)

modeloRedeNeural1 = Sequential()
modeloRedeNeural1.add(Dense(8,activation="relu",input_shape=(9,)))
modeloRedeNeural1.add(Dense(8,activation="relu"))
modeloRedeNeural1.add(Dense(8,activation="relu"))
modeloRedeNeural1.add(Dense(4,activation="softmax"))

modeloRedeNeural1.compile(optimizer="adam",
                         loss="categorical_crossentropy",
                         metrics=["accuracy",
                                  tf.keras.metrics.Precision(name='precision'),  # Precisão
                                  tf.keras.metrics.Recall(name='recall'),
                                  tf.keras.metrics.F1Score(name='f1_score')]
                          )

modeloRedeNeural2 = Sequential()
modeloRedeNeural2.add(Dense(4,activation="relu",input_shape=(9,)))
modeloRedeNeural2.add(Dense(4,activation="softmax"))

modeloRedeNeural2.compile(optimizer="adam",
                         loss="categorical_crossentropy",
                         metrics=["accuracy",
                                  tf.keras.metrics.Precision(name='precision'),  # Precisão
                                  tf.keras.metrics.Recall(name='recall'),
                                  tf.keras.metrics.F1Score(name='f1_score')])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
historico1 = modeloRedeNeural1.fit(dadosTreinoEntradaNormalizados,
                                 dadosTreinoSaida,
                                 epochs=24,
                                 batch_size=32,
                                 validation_split=0.20,
                                 verbose=1)
historico2 = modeloRedeNeural2.fit(dadosTreinoEntradaNormalizados,
                                 dadosTreinoSaida,
                                 epochs=50,
                                 batch_size=32,
                                 validation_split=0.20,
                                 verbose=1)

Epoch 1/24
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.2297 - f1_score: 0.1652 - loss: 1.3338 - precision: 0.8750 - recall: 0.0109 - val_accuracy: 0.2562 - val_f1_score: 0.1848 - val_loss: 1.3151 - val_precision: 1.0000 - val_recall: 0.0500
Epoch 2/24
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4281 - f1_score: 0.3249 - loss: 1.2277 - precision: 1.0000 - recall: 0.1172 - val_accuracy: 0.4375 - val_f1_score: 0.3610 - val_loss: 1.2168 - val_precision: 0.9524 - val_recall: 0.1250
Epoch 3/24
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5344 - f1_score: 0.4359 - loss: 1.1285 - precision: 0.9821 - recall: 0.1719 - val_accuracy: 0.5063 - val_f1_score: 0.4295 - val_loss: 1.1278 - val_precision: 0.9630 - val_recall: 0.1625
Epoch 4/24
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6062 - f1_score: 0.5356 - loss: 1.0293 - precision: 0.9865 - recall: 0.2281 - val_accuracy: 0.5688 - val_f1_score: 0.5202 - val_loss: 1.0327 - val_precision: 0.9722 - val_recall: 0.2

In [ ]:
metrics1 = modeloRedeNeural1.evaluate(
    dadosTesteEntradaNormalizados,
    dadosTesteSaida,
    verbose=0, return_dict=True)

perda1 = metrics1["loss"]
acuracia1 = metrics1["accuracy"]
f1score1 = metrics1["f1_score"]
recall1 = metrics1["recall"]

metrics2 = modeloRedeNeural2.evaluate(
    dadosTesteEntradaNormalizados,
    dadosTesteSaida,
    verbose=0, return_dict=True)

perda2 = metrics2["loss"]
acuracia2 = metrics2["accuracy"]
f1score2 = metrics2["f1_score"]
recall2 = metrics2["recall"]


In [ ]:
print(f"Erro do modelo (Loss): {perda1:.4f}")
print(f"Taxa de acerto (Accuracy): {acuracia1:.4%}")
print(f"Erro do modelo (f1): {f1score1[0]:.4f}")
print(f"Taxa de acerto (recall): {recall1:.4%}")
print(f"Erro do modelo (Loss): {perda2:.4f}")
print(f"Taxa de acerto (Accuracy): {acuracia2:.4%}")
print(f"Erro do modelo (f1): {f1score2[0]:.4f}")
print(f"Taxa de acerto (recall): {recall2:.4%}")

Erro do modelo (Loss): 0.0337
Taxa de acerto (Accuracy): 100.0000%
Erro do modelo (f1): 1.0000
Taxa de acerto (recall): 99.5000%
Erro do modelo (Loss): 0.1380
Taxa de acerto (Accuracy): 99.0000%
Erro do modelo (f1): 0.9804
Taxa de acerto (recall): 97.0000%


In [ ]:
novoPaciente1 = pd.DataFrame({
    "idade": [58],
    "hemoglobina": [14.2],
    "hematocrito": [42],
    "hemacias": [4.8],
    "leucocitos": [7000],
    "plaquetas": [260],
    "glicemia": [210],
    "pressaoSistolica": [120],
    "pressaoDiastolica": [80]
    })

novoPaciente2 = pd.DataFrame({
    "idade": [58],
    "hemoglobina": [8.5],
    "hematocrito": [26.0],
    "hemacias": [2.9],
    "leucocitos": [85000],
    "plaquetas": [45],
    "glicemia": [210],
    "pressaoSistolica": [120],
    "pressaoDiastolica": [80]
})

# 1. Muito próximo de ser saudável, mas tem leucemia
# Todos os parâmetros clínicos e pressóricos estão normais, exceto a contagem de leucócitos (muito elevada)
paciente_leucemia = pd.DataFrame({
    "idade": [42],
    "hemoglobina": [13.8],
    "hematocrito": [41.5],
    "hemacias": [4.6],
    "leucocitos": [51000],          # Leucocitose acentuada
    "plaquetas": [150],             # Plaquetas discretamente reduzidas ou normais em milhares (140.000)
    "glicemia": [88],
    "pressaoSistolica": [118],
    "pressaoDiastolica": [78]
})

# 2. Muito próximo de ser saudável, mas tem hipertensão
# Hemograma e glicemia perfeitamente saudáveis, porém com pressão em estágio de hipertensão
paciente_hipertenso = pd.DataFrame({
    "idade": [45],
    "hemoglobina": [14.5],
    "hematocrito": [43.0],
    "hemacias": [4.8],
    "leucocitos": [6500],
    "plaquetas": [250],
    "glicemia": [90],
    "pressaoSistolica": [135],       # Hipertensão estágio 1 (>= 140 mmHg)
    "pressaoDiastolica": [90]        # Hipertensão estágio 1 (>= 90 mmHg)
})

# 3. Muito próximo de ser hipertenso, mas é saudável
# Hemograma e glicemia normais com pressão na faixa de pré-hipertensão/pré-pressão alta (ainda considerado saudável/não hipertenso)
paciente_limitrofe_saudavel = pd.DataFrame({
    "idade": [40],
    "hemoglobina": [14.2],
    "hematocrito": [42.0],
    "hemacias": [4.7],
    "leucocitos": [6800],
    "plaquetas": [260],
    "glicemia": [89],
    "pressaoSistolica": [138],       # Pré-hipertensão (120–129 mmHg)
    "pressaoDiastolica": [84]        # Pré-hipertensão (80–84 mmHg)
})


In [ ]:

def preverDiagnostico(novoPaciente, modeloRedeNeural):
  novoPacienteNormalizado = normalizador.transform(novoPaciente)
  probabilidades = modeloRedeNeural.predict(novoPacienteNormalizado)
  indiceClasse = probabilidades.argmax()
  diagnostico = encoderDiagnostico.inverse_transform([indiceClasse])
  previsions = []
  print("Diagnóstico previsto:", diagnostico[0])

  print("\nProbabilidades:")

  for i in range(len(encoderDiagnostico.classes_)):
      previsions.append(round(probabilidades[0][i] * 100, 2))
      print(encoderDiagnostico.classes_[i],":",round(probabilidades[0][i] * 100, 2),"%")
  print("\n")
  return previsions

In [ ]:
prev_list = [preverDiagnostico(novoPaciente1, modeloRedeNeural1),
             preverDiagnostico(novoPaciente2, modeloRedeNeural1),
             preverDiagnostico(paciente_leucemia, modeloRedeNeural1),
             preverDiagnostico(paciente_hipertenso, modeloRedeNeural1),
             preverDiagnostico(paciente_limitrofe_saudavel, modeloRedeNeural1),
             preverDiagnostico(novoPaciente1, modeloRedeNeural2),
             preverDiagnostico(novoPaciente2, modeloRedeNeural2),
             preverDiagnostico(paciente_leucemia, modeloRedeNeural2),
             preverDiagnostico(paciente_hipertenso, modeloRedeNeural2),
             preverDiagnostico(paciente_limitrofe_saudavel, modeloRedeNeural2)]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Diagnóstico previsto: Diabetes

Probabilidades:
Diabetes : 99.96 %
Hipertensao : 0.01 %
Leucemia : 0.01 %
Saudavel : 0.02 %


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
Diagnóstico previsto: Leucemia

Probabilidades:
Diabetes : 0.0 %
Hipertensao : 0.0 %
Leucemia : 100.0 %
Saudavel : 0.0 %


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Diagnóstico previsto: Saudavel

Probabilidades:
Diabetes : 0.84 %
Hipertensao : 2.3 %
Leucemia : 8.25 %
Saudavel : 88.61 %


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
Diagnóstico previsto: Saudavel

Probabilidades:
Diabetes : 2.05 %
Hipertensao : 27.01 %
Leucemia : 1.95 %
Saudavel : 68.99 %


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Diagnóstico previsto: Saudavel

Probabilidades:
Diabetes : 2.83 %
Hipertensao : 17.1 %
Leucemia : 2.5 %
Saudavel : 77.56 %


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
Diagnóstico previsto: Diabetes

Probabilidades:
Diabetes : 97.91 %
Hipertensao : 0.03 %
Leucemia : 1.08 %
Saudavel : 0.99 %


1/1 ━━━━━━━━━━━━━━━━

In [ ]:
prev_df = pd.DataFrame(prev_list, columns=encoderDiagnostico.classes_)
prev_df

In [ ]:
modeloRedeNeural1.save("99acc98f199rec.h5")
modeloRedeNeural2.save("98acc96f197rec.h5")

In [ ]:
import joblib

joblib.dump(normalizador, "98acc96f197rec.scaler.pkl")
joblib.dump(normalizador, "99acc98f199rec.scaler.pkl")

['98acc96f197rec.scaler.pkl']